# NB 05 · LLM misjudgement

**Series:** Learning LangGraph through literate notebooks, notebook 5 of 7.
**Central question:** where do the rules and the model's own judgement disagree, and what does that say about rule-based checks?

Since NB 03 the work has been split: the model *reads* (one figure per document, or a search query),
and fixed rules *judge* (which sources count, whether they contradict each other, whether the evidence
is enough). This notebook asks what happens if the model judges instead. It gives the model the same
documents, asks for a verdict in a fixed form, and compares that verdict field by field with the rules.

## Learning goals

After this notebook you can:

- ask a model for a structured judgement of evidence and compare it, field by field, with a rule-based assessment;
- recognise three misjudgements observed here: a missed contradiction, confidence resting on one source, and copies taken for independent confirmation;
- keep three properties apart that a model blurred: a contradiction exists, it is resolved, the evidence is sufficient;
- judge what metadata, a different prompt and a "thinking" mode did and did not change;
- state the limits of such a test: one model, one prompt, one corpus.

## Setup

Every verdict in this notebook was recorded from `qwen3:8b` (temperature 0, thinking off) on
5 October 2026; the scripted mode replays those recordings. The misjudgements shown below are the
model's own. Where a case had to be constructed, the notebook says so.

In [ ]:
import json
import re
import sys
from importlib.metadata import version

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

from research_assistant.assistant import corpus, extract_claim, weights
from research_assistant.evidence import CANONICAL_QUESTIONS, assess, origin
from research_assistant.judgement import LISTING_JUDGE_PROMPT, Verdict
from research_assistant.models import get_chat_model, get_llm_mode
from research_assistant.retrieval import retrieve
from research_assistant.state import SourceDoc

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")

mode = get_llm_mode()
if mode == "disabled":
    mode = "fake"  # this notebook needs a model; the recorded verdicts stand in
llm = get_chat_model(mode, fixture=("nb03_claims.yaml", "nb05_judgements.yaml"))
print(f"Model mode: {mode}")

QUESTIONS = CANONICAL_QUESTIONS
BY_ID = {doc["source_id"]: doc for doc in corpus()}

## Concept: reading is not judging

Reading asks: *what does this document say?* Judging asks: *what follows from all the documents
together?* The second question needs weighing: how authoritative each source is, whether two sources
are really one, whether a figure has been replaced, whether disagreement is about facts or opinions.
The rules of NB 03 do that weighing in a fixed, visible way. A model can do it too, in fluent prose.
The question is whether its weighing holds up.

To compare like with like, the model fills in the same five fields the rules produce:

| Field | Rule that decides it |
|---|---|
| `answer` | the one figure left after P1, P3 and P4 |
| `contradiction` | P3: documents state incompatible figures, always flagged |
| `independent_sources` | independence: a copy counts once with its original |
| `outdated` | P4: a newer source on the same point, at least 12 months younger |
| `sufficient` | P2: at least two independent sources at or above the floor, summed authority of at least 1.0 |

**The limits of this test, stated before any result:** one model (`qwen3:8b`), one prompt in two
versions, one small corpus, temperature 0. What follows is a documented case of where *this* model
disagreed with *these* rules, not a claim about how every language model behaves.

## Experiment

### 1 The judge prompt and the comparison

The model sees the question and the documents, each with its metadata: publisher, type, date,
authority weight and, for a copy, the document it is based on. It replies in a fixed JSON form.

In [ ]:
JUDGE_PROMPT = ChatPromptTemplate.from_template(
    "You are assessing evidence for a research question. Read the documents and judge the evidence.\n"
    "Reply with JSON only, in exactly this form:\n"
    '{{"answer": "<the figure you would report, or null>", "contradiction": <true if the documents state '
    'incompatible figures, else false>, "independent_sources": <number of independent sources that support '
    'your answer>, "outdated": [<ids of documents that are outdated>], "sufficient": <true if the evidence is '
    'strong enough to answer with confidence, else false>, "reason": "<one sentence>"}}\n\n'
    "Question: {question}\n\nDocuments:\n{documents}"
)


def format_documents(documents: list[SourceDoc], with_metadata: bool = True) -> str:
    blocks = []
    for doc in documents:
        block = f"[{doc['source_id']}] {doc['title']}\n"
        if with_metadata:
            meta = (f"publisher: {doc['publisher']}; type: {doc['type']}; "
                    f"date: {doc['date']}; authority: {doc['authority_weight']}")
            if doc["based_on"]:
                meta += f"; based on: {', '.join(doc['based_on'])}"
            block += f"({meta})\n"
        blocks.append(block + doc["text"])
    return "\n\n".join(blocks)


def read_verdict(reply: str) -> Verdict:
    text = reply.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    data = json.loads(text)
    answer = data.get("answer")
    return Verdict(
        answer=None if answer in (None, "null") else str(answer),
        contradiction=bool(data["contradiction"]),
        independent_sources=int(data["independent_sources"]),
        outdated=sorted(data.get("outdated") or []),
        sufficient=bool(data["sufficient"]),
        reason=str(data.get("reason", "")),
    )


def judge(question, documents, with_metadata=True, prompt=JUDGE_PROMPT) -> Verdict:
    messages = prompt.invoke({"question": question, "documents": format_documents(documents, with_metadata)})
    return read_verdict(llm.invoke(messages).content)

The comparison takes the model's verdict and the rules' assessment and lines them up field by field.
For `independent_sources`, the rules count how many independent origins state the figure the model
chose, whatever their authority, so that the question is purely about independence.

In [ ]:
def number(figure: str | None) -> float | None:
    match = re.search(r"\d+(?:\.\d+)?", figure or "")
    return float(match.group()) if match else None


def origins_stating(figure, documents, claims) -> int:
    by_id = {doc["source_id"]: doc for doc in documents}
    target = number(figure)
    ids = [sid for sid, claim in claims.items() if sid in by_id and target is not None and number(claim) == target]
    return len({origin(sid, by_id) for sid in ids})


def compare(verdict, assessment, documents, claims):
    outdated = sorted(sid for sid, reason in assessment.disregarded.items() if reason.startswith("outdated"))
    origins = origins_stating(verdict.answer, documents, claims)
    return [
        ("answer", verdict.answer, assessment.answer, number(verdict.answer) == number(assessment.answer)),
        ("contradiction", verdict.contradiction, assessment.contradiction_flag, verdict.contradiction == assessment.contradiction_flag),
        ("independent sources for that answer", verdict.independent_sources, origins, verdict.independent_sources == origins),
        ("outdated", verdict.outdated, outdated, verdict.outdated == outdated),
        ("sufficient", verdict.sufficient, assessment.sufficient, verdict.sufficient == assessment.sufficient),
    ]


def side_by_side(question: str, documents: list[SourceDoc], **judge_options):
    claims = {doc["source_id"]: extract_claim(llm, question, doc) for doc in documents}
    print("Documents:")
    for doc in documents:
        copy = f", based on {doc['based_on'][0]}" if doc["based_on"] else ""
        print(f"  {doc['source_id']}  {doc['date']}  authority {doc['authority_weight']:<4}  claims {claims[doc['source_id']]}{copy}")
    assessment = assess(documents, claims)
    print(f"\nRules: answer {assessment.answer}, contradiction {assessment.contradiction_flag}, "
          f"resolved {assessment.contradiction_resolved}, sufficient {assessment.sufficient}")
    verdict = judge(question, documents, **judge_options)
    print(f'Model: answer {verdict.answer}, contradiction {verdict.contradiction}, sufficient {verdict.sufficient}')
    print(f'       reason: "{verdict.reason}"\n')
    rows = compare(verdict, assessment, documents, claims)
    for field, model_value, rules_value, agree in rows:
        print(f"  {'agree  ' if agree else 'DIFFERS'}  {field:<36} model {model_value!s:<14} rules {rules_value}")
    return verdict, assessment, claims, rows

### 2 Topic b: where model and rules agree

Three reputable sources, one figure, different opinions about the merger. The control case comes
first, so that the disagreements below are not mistaken for a broken comparison.

In [ ]:
_ = side_by_side(QUESTIONS["b"], retrieve(QUESTIONS["b"], corpus(), weights()))

Full agreement, including on the point most likely to trip a model: the sources' opinions differ,
their facts do not, and the model did not call that a contradiction.

### 3 Topic a: the missed contradiction

In [ ]:
verdict_a, assessment_a, claims_a, rows_a = side_by_side(QUESTIONS["a"], retrieve(QUESTIONS["a"], corpus(), weights()))

The model's answer is right, and its reason sounds careful. But it reports `contradiction: false`,
and its reason does not mention A3 at all: the forum post that claims CHF 310 million. The rules flag
the contradiction (P3: always flagged) and resolve it, because the only source of the lower figure is
below the authority floor. The difference matters beyond this question: an answer that silently drops
a dissenting claim cannot be checked by its reader. With the same document read on its own in NB 03,
the model reported A3's figure correctly; it was in judging all documents at once that A3 disappeared.

### 4 Topic d: recency right, contradiction blurred

In [ ]:
_ = side_by_side(QUESTIONS["d"], retrieve(QUESTIONS["d"], corpus(), weights()))

The plan for this notebook expected the model to take the 2023 target of 60% for the current one.
It did not: it chose 45%, named D1 and D4 as outdated and called the evidence sufficient, exactly as
the rules do. Recency, here, was not a weakness of this model.

The one difference is `contradiction`. The model says there is none; the rules say there is one (60%
against 45%) and that it is resolved, because the older side is outdated. The model folded "resolved"
into "absent". NB 03 kept these apart on purpose:

- **a contradiction exists:** documents state incompatible figures;
- **the contradiction is resolved:** the rules still leave one supported figure;
- **the evidence is sufficient:** the quality gate is passed.

An assessment that reports "no contradiction" here hides that the answer depends on recognising an
outdated source, which is precisely the step a reader would want to check.

### 5 Topic e: confidence resting on one source

In [ ]:
verdict_e, assessment_e, claims_e, rows_e = side_by_side(QUESTIONS["e"], retrieve(QUESTIONS["e"], corpus(), weights()))

The model picks the right figure, 0.62%, and even counts the sources correctly: one. Then it calls
the evidence **sufficient**. The rules do not (P2: at least two independent qualifying sources), which
is why NB 04 had to search again. The model also reports no contradiction, although 0.38% and 0.62%
stand side by side. Its reason, "provides the most direct and specific information", is the kind of
plausible sentence that makes overconfidence hard to spot.

### 6 A constructed case: only the copies were found

What if round 1 had found only E1 and E2, the blog and the newsletter that copies it? This case is
constructed, the same way NB 06 will construct a contradiction between credible sources. The model
still sees the metadata, including E2's line "based on: E1".

In [ ]:
verdict_copies, assessment_copies, claims_copies, rows_copies = side_by_side(QUESTIONS["e"], [BY_ID["E1"], BY_ID["E2"]])

The model reports 0.38%, **two independent sources** and sufficient evidence, and its reason says the
two documents "independently report" the figure. The metadata right in front of it says that E2 is
based on E1. The rules count one origin, find both documents below the authority floor, and give no
answer. Source count is not source independence.

### 7 Does the metadata make a difference?

Every verdict above was given with the metadata in view. The same prompt without any metadata (only
title and text) gave the same disagreements in every case:

In [ ]:
for name, question, documents in (
    ("a", QUESTIONS["a"], retrieve(QUESTIONS["a"], corpus(), weights())),
    ("e", QUESTIONS["e"], retrieve(QUESTIONS["e"], corpus(), weights())),
    ("copies", QUESTIONS["e"], [BY_ID["E1"], BY_ID["E2"]]),
):
    claims = {doc["source_id"]: extract_claim(llm, question, doc) for doc in documents}
    assessment = assess(documents, claims)
    for with_metadata in (True, False):
        verdict = judge(question, documents, with_metadata=with_metadata)
        differs = [field for field, _, _, agree in compare(verdict, assessment, documents, claims) if not agree]
        print(f"{name:<7} metadata {str(with_metadata):<5} differs on {differs}")

Giving the model the information the rules use did not make it use that information. Authority
weights and the "based on" link were there; the verdicts did not change.

## Common mistakes

These mistakes are not in the code but in how one would use the model's verdict. Each cell shows the
recorded verdict next to the rule it breaks. In scripted mode the cell asserts the recorded
disagreement; with a live model it prints what the model said this time, because a live model may
answer differently, and that variation is part of the finding.

### Mistake 1: trusting the model's "no contradiction"

In [ ]:
print(f"model: contradiction {verdict_a.contradiction} | rules: contradiction {assessment_a.contradiction_flag}")
if mode == "fake":
    assert verdict_a.contradiction is False and assessment_a.contradiction_flag is True

A pipeline that asked the model whether sources conflict, and moved on when it said no, would never
show a reader the dissenting CHF 310 million. **Fix:** detect contradictions by comparing the figures
each document states, as the rules do after the model has read each document separately.

### Mistake 2: treating the model's "sufficient" as the quality gate

In [ ]:
print(f"model: sufficient {verdict_e.sufficient} with {verdict_e.independent_sources} source | rules: sufficient {assessment_e.sufficient}")
if mode == "fake":
    assert verdict_e.sufficient is True and assessment_e.sufficient is False

Had the router of NB 04 read the model's `sufficient` instead of the evidence score, topic e would
have been answered in round 1 from a single source, and E4 would never have been searched for.
**Fix:** the gate is a rule with stated thresholds (P2), not a model's sense of confidence.

### Mistake 3: counting agreeing copies as independent confirmation

In [ ]:
print(f"model: {verdict_copies.independent_sources} independent sources | rules: "
      f"{origins_stating(verdict_copies.answer, [BY_ID['E1'], BY_ID['E2']], claims_copies)} origin")
if mode == "fake":
    assert verdict_copies.independent_sources == 2
    assert origins_stating(verdict_copies.answer, [BY_ID["E1"], BY_ID["E2"]], claims_copies) == 1

Two documents that agree look like corroboration. Here they are one claim, copied. **Fix:** follow
the provenance (`based_on`) and count origins, not documents, as the independence rule does.

## Prompt log

The verdicts above use the first version of the judge prompt. Two changes were tried, both recorded
from `qwen3:8b` on 5 October 2026.

**Version 2: list every figure first.** The prompt asks the model to list, for every document, the
figure it states, and only then to judge; it also defines independence ("a document based on
another counts once with it").

In [ ]:
for name, question, documents in (
    ("a", QUESTIONS["a"], retrieve(QUESTIONS["a"], corpus(), weights())),
    ("d", QUESTIONS["d"], retrieve(QUESTIONS["d"], corpus(), weights())),
    ("e", QUESTIONS["e"], retrieve(QUESTIONS["e"], corpus(), weights())),
    ("copies", QUESTIONS["e"], [BY_ID["E1"], BY_ID["E2"]]),
):
    claims = {doc["source_id"]: extract_claim(llm, question, doc) for doc in documents}
    assessment = assess(documents, claims)
    for label, prompt in (("version 1", JUDGE_PROMPT), ("version 2", LISTING_JUDGE_PROMPT)):
        verdict = judge(question, documents, prompt=prompt)
        differs = [field for field, _, _, agree in compare(verdict, assessment, documents, claims) if not agree]
        print(f"{name:<7} {label}: differs on {differs or 'nothing'}")

**What happened.** Version 2 fixed topic e completely: it now flagged the contradiction and called one
source insufficient. But topic d, which version 1 had judged correctly on sufficiency, turned
**over-cautious**: the model now called the evidence insufficient because of the (resolved)
contradiction. Topic a stayed wrong, and the model's own list gave the forum post's figure as `null`,
although the same model, reading A3 on its own in NB 03, reported CHF 310 million. For the copies,
the model now counted one source, and still called the evidence sufficient, with the reason "multiple
independent sources confirm": the verdict contradicts itself.

**Thinking mode.** Version 1 on topic a with qwen3's thinking switched on (`OLLAMA_REASONING=true`)
caught the contradiction: *"Two independent sources (press release and news article) report CHF 342
million revenue for Q3 2025, while a forum post with lower authority contradicts this figure."* It
took 91 seconds instead of 15. The next cell repeats this live when `LLM_MODE=ollama`; it is skipped in
scripted mode.

In [ ]:
from research_assistant.models import get_chat_model as _get_model
import os

os.environ["OLLAMA_REASONING"] = "true"
thinking_llm = _get_model("ollama")
os.environ["OLLAMA_REASONING"] = "false"
documents = retrieve(QUESTIONS["a"], corpus(), weights())
reply = thinking_llm.invoke(JUDGE_PROMPT.invoke({"question": QUESTIONS["a"], "documents": format_documents(documents)})).content
print(read_verdict(reply))

**Insight.** Rewording the prompt moved the errors around instead of removing them: one case fixed,
one case newly broken, one unchanged. Thinking mode fixed the case it was tried on, at six times the
cost, with no guarantee for the others. What reliably helped was the structure of NB 03 rather than any prompt:
the model reads one document at a time and reports one figure, and fixed rules compare the
figures. That is the practical answer to research question 4. Rule-based checks have blind spots of
their own (hand-assigned weights, a measured threshold with narrow margins), but they fail the same way
every time and can be inspected; the model's judgement shifted with each change of wording.

## Summary

Given the same documents, `qwen3:8b` agreed with the rules on the control case and on recency, and
disagreed in three ways: it missed a contradiction, it was confident on a single source, and it took
two copies for independent confirmation even with the "based on" link in view. It also blurred a
resolved contradiction into no contradiction. Metadata did not change its verdicts; a reworded prompt
changed which cases it got wrong; thinking mode fixed one case at six times the cost.

The boundary stays as stated at the start: this documents one model, one prompt in two versions, one
corpus. It shows why the assistant lets the model read and the rules judge, and why every rule is
written down where it can be checked.

The judge and the comparison live in the package as `judgement.py`. The next cell checks that the
notebook and the package agree; it compares the parsing and the comparison in every mode and complete
model calls only in scripted mode.

In [ ]:
from research_assistant import judgement as package_judgement

assert package_judgement.JUDGE_PROMPT == JUDGE_PROMPT
documents_a = retrieve(QUESTIONS["a"], corpus(), weights())
assert package_judgement.format_documents(documents_a) == format_documents(documents_a)
assert package_judgement.compare(verdict_a, assessment_a, documents_a, claims_a) == rows_a
sample = '{"answer": 0.38, "contradiction": false, "independent_sources": 2, "outdated": [], "sufficient": true, "reason": "x"}'
assert package_judgement.read_verdict(sample) == read_verdict(sample)
if mode == "fake":
    assert package_judgement.judge(llm, QUESTIONS["a"], documents_a) == verdict_a
print("Notebook and package agree.")

**Next: NB 06, persistence.** The assistant can now answer, search again or stop with a limitation.
NB 06 lets it pause instead: when the evidence stays uncertain, the run saves its state, waits for a
person to decide, and continues where it stopped.

## Exercise

1. Lower the authority floor (P1) from 0.5 to 0.3 and assess topic e again with the rules
   (`assess` uses `research_assistant.evidence.AUTHORITY_FLOOR`). Work out first what should happen to
   E1, E2 and the contradiction, then check. Does the rule change bring the rules closer to the
   model's verdict, or further away? What does that say about who should set such thresholds?
2. With `LLM_MODE=ollama`, write a third version of the judge prompt that you expect to fix topic a,
   and test it on topics a and d. Did it fix a without breaking d?